In [ ]:
!sudo apt-get install -y zstd
!pip install dspy-ai

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import threading
import subprocess

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
!ollama pull gemma3:1b
!ollama pull qwen2.5:3b
!ollama pull phi3:mini
!ollama pull mistral:7b-instruct-q4_0

In [ ]:
import dspy
import requests
import sqlite3
import time

In [ ]:
def setup_ollama(model_name):
    """Verifica se o Ollama está rodando e exibe o modelo ativo."""
    try:
        requests.get("http://localhost:11434/api/tags")
        print(f"Ollama rodando - {model_name}")
        return True
    except:
        print("Ollama não está rodando.")
        return False

In [ ]:
def create_streaming_database():
    conn = sqlite3.connect("streaming.db")
    c = conn.cursor()

    c.execute("DROP TABLE IF EXISTS reviews")
    c.execute("DROP TABLE IF EXISTS views")
    c.execute("DROP TABLE IF EXISTS users")
    c.execute("DROP TABLE IF EXISTS movies")

    c.execute("""
        CREATE TABLE movies (
            id INTEGER PRIMARY KEY,
            title TEXT,
            genre TEXT,
            release_year INTEGER,
            director TEXT,
            duration_min INTEGER,
            rating REAL
        )
    """)

    c.execute("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            name TEXT,
            country TEXT,
            plan TEXT,
            join_date TEXT
        )
    """)

    c.execute("""
        CREATE TABLE views (
            id INTEGER PRIMARY KEY,
            user_id INTEGER,
            movie_id INTEGER,
            watch_date TEXT,
            completed INTEGER
        )
    """)

    c.execute("""
        CREATE TABLE reviews (
            id INTEGER PRIMARY KEY,
            user_id INTEGER,
            movie_id INTEGER,
            score INTEGER,
            comment TEXT
        )
    """)

    c.executemany("INSERT INTO movies VALUES (?, ?, ?, ?, ?, ?, ?)", [
        (1,  "Parasite",                  "Thriller", 2019, "Bong Joon-ho",       130, 8.5),
        (2,  "Oldboy",                    "Thriller", 2003, "Park Chan-wook",      120, 8.4),
        (3,  "Train to Busan",            "Action",   2016, "Yeon Sang-ho",        118, 7.6),
        (4,  "A Tale of Two Sisters",     "Horror",   2003, "Kim Jee-woon",        115, 7.8),
        (5,  "The Wailing",               "Horror",   2016, "Na Hong-jin",         156, 7.9),
        (6,  "Burning",                   "Drama",    2018, "Lee Chang-dong",      148, 7.5),
        (7,  "Memories of Murder",        "Crime",    2003, "Bong Joon-ho",        132, 8.1),
        (8,  "Twinkling Watermelon",      "Romance",  2023, "Son Jeong-hyun",      140, 8.3),
        (9,  "My Sassy Girl",             "Romance",  2001, "Kwak Jae-yong",       137, 7.8),
        (10, "The Man from Nowhere",      "Action",   2010, "Lee Jeong-beom",      119, 7.8),
    ])

    c.executemany("INSERT INTO users VALUES (?, ?, ?, ?, ?)", [
        (1, "Marta",   "Brazil", "Premium", "2022-01-10"),
        (2, "Emily",   "USA",    "Basic",   "2021-05-20"),
        (3, "Wendy",   "Brazil", "Premium", "2023-03-15"),
        (4, "Webster", "Canada", "Basic",   "2022-08-01"),
        (5, "Ryan",    "Brazil", "Premium", "2020-11-25"),
    ])

    c.executemany("INSERT INTO views VALUES (?, ?, ?, ?, ?)", [
        (1,  1, 1,  "2024-01-05", 1),
        (2,  1, 3,  "2024-01-10", 1),
        (3,  2, 5,  "2024-01-12", 0),
        (4,  3, 2,  "2024-02-01", 1),
        (5,  4, 7,  "2024-02-10", 1),
        (6,  5, 4,  "2024-03-05", 1),
        (7,  1, 6,  "2024-03-10", 1),
        (8,  2, 8,  "2024-03-15", 0),
        (9,  3, 9,  "2024-04-01", 1),
        (10, 5, 10, "2024-04-10", 1),
    ])

    c.executemany("INSERT INTO reviews VALUES (?, ?, ?, ?, ?)", [
        (1, 1, 1,  9,  "Crítica social muito boa!"),
        (2, 1, 3,  10, "adorei!"),
        (3, 2, 5,  8,  "Assustador"),
        (4, 3, 2,  10, "Clássico do cinema coreano"),
        (5, 4, 7,  9,  "Investigação incrível"),
        (6, 5, 4,  9,  "Me deixou com medo por dias"),
    ])

    conn.commit()
    conn.close()
    print("Banco de dados de streaming criado")

create_streaming_database()

In [ ]:
class GenerateSQL(dspy.Signature):
    """
    Gera queries SQL a partir de linguagem natural.

    Schema do banco de dados de streaming:
    - movies: id, title, genre, release_year, director, duration_min, rating
    - users: id, name, country, plan, join_date
    - views: id, user_id, movie_id, watch_date, completed
    - reviews: id, user_id, movie_id, score, comment
    """

    question: str = dspy.InputField(desc="Pergunta em linguagem natural sobre a plataforma de streaming")

    sql_query: str = dspy.OutputField(desc="Query SQL válida para responder a pergunta")


class RefineSQL(dspy.Signature):
    """Refina uma query SQL que gerou erro."""

    question:  str = dspy.InputField(desc="Pergunta original do usuário")
    sql_query: str = dspy.InputField(desc="Query SQL que falhou")
    error:     str = dspy.InputField(desc="Mensagem de erro retornada")

    refined_sql: str = dspy.OutputField(desc="Query SQL corrigida e válida")

print("Signatures definidas")

In [ ]:
class SQLGeneratorModule(dspy.Module):
    """
    Módulo principal de geração de SQL.
    Usa ChainOfThought multichain: geração + refinamento em cadeia.
    Conexão com o banco é feita internamente para compatibilidade com otimizador.
    """

    def __init__(self):
        super().__init__()

        self.generator = dspy.ChainOfThought(GenerateSQL)

        self.refiner   = dspy.ChainOfThought(RefineSQL)

    def forward(self, question):
        db = sqlite3.connect("streaming.db")

        output = self.generator(question=question)
        sql = output.sql_query.strip()

        if "```" in sql:
            sql = sql.split("```")[1].replace("sql", "").strip()

        try:
            results = db.execute(sql).fetchall()
            db.close()
            return dspy.Prediction(sql_query=sql, results=results, error=None)

        except Exception as e:
            refined = self.refiner(question=question, sql_query=sql, error=str(e))
            refined_sql = refined.refined_sql.strip()

            if "```" in refined_sql:
                refined_sql = refined_sql.split("```")[1].replace("sql", "").strip()

            try:
                results = db.execute(refined_sql).fetchall()
                db.close()
                return dspy.Prediction(sql_query=refined_sql, results=results, error=None)
            except Exception as e2:
                db.close()
                return dspy.Prediction(sql_query=refined_sql, results=None, error=str(e2))

print("Módulo SQLGeneratorModule definido")

In [ ]:
def create_examples():
    return [
        dspy.Example(
            question="Quantos filmes de Horror existem na plataforma?",
            sql_query="SELECT COUNT(*) FROM movies WHERE genre = 'Horror'"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual o filme com maior rating?",
            sql_query="SELECT title, rating FROM movies ORDER BY rating DESC LIMIT 1"
        ).with_inputs("question"),

        dspy.Example(
            question="Liste todos os usuários do Brasil com plano Premium",
            sql_query="SELECT name, plan FROM users WHERE country = 'Brazil' AND plan = 'Premium'"
        ).with_inputs("question"),

        dspy.Example(
            question="Quantas visualizações completas foram registradas?",
            sql_query="SELECT COUNT(*) FROM views WHERE completed = 1"
        ).with_inputs("question"),

        dspy.Example(
            question="Qual a média de score das avaliações?",
            sql_query="SELECT AVG(score) FROM reviews"
        ).with_inputs("question"),

        dspy.Example(
            question="Quais filmes foram dirigidos por Bong Joon-ho?",
            sql_query="SELECT title, release_year FROM movies WHERE director = 'Bong Joon-ho'"
        ).with_inputs("question"),
    ]

examples = create_examples()
print(f"{len(examples)} exemplos criados")

In [ ]:
def build_optimized_module():
    """
    Aplica otimização com LabeledFewShot.
    Seleciona automaticamente os melhores exemplos para o módulo.
    """
    base_module = SQLGeneratorModule()

    optimizer = dspy.LabeledFewShot(k=4)

    optimized = optimizer.compile(
        student=base_module,
        trainset=examples
    )

    print("Módulo otimizado com LabeledFewShot")
    return optimized

In [ ]:
test_questions = [
    "Quantos filmes de ação existem na plataforma?",
    "Qual a duração média dos filmes?",
    "Liste os filmes lançados após 2010 com rating acima de 7.8",
    "Quais usuários têm plano Premium?",
    "Quais filmes a Marta assistiu?",
]

print("Perguntas de teste:")
for i, q in enumerate(test_questions, 1):
    print(f"  {i}. {q}")

In [ ]:
import os

MODELS = [
    "gemma3:1b",
    "qwen2.5:3b",
    "phi3:mini",
    "mistral:7b-instruct-q4_0",
]

def ensure_ollama_running():
    """Garante que o Ollama está rodando, reiniciando se necessário."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        if r.status_code == 200:
            return True
    except:
        pass

    print("  Reiniciando Ollama...")
    subprocess.Popen(["ollama", "serve"])
    time.sleep(5)

    try:
        requests.get("http://localhost:11434/api/tags", timeout=3)
        print("Ollama reiniciado com sucesso")
        return True
    except:
        print("Falha ao reiniciar Ollama")
        return False

def configure_model(model_name):
    """Configura o DSPy para usar um modelo específico do Ollama."""
    dspy.settings.configure(
        lm=dspy.LM(
            model=f"ollama/{model_name}",
            api_base="http://localhost:11434",
            max_tokens=500,
            temperature=0.7,
        )
    )

def test_model(model_name, questions):
    """Testa todas as perguntas com um modelo e coleta métricas."""
    print(f"\nMODELO: {model_name}")

    ensure_ollama_running()
    configure_model(model_name)
    module = build_optimized_module()
    results_log = []

    for i, question in enumerate(questions, 1):
        print(f"\n{i}: {question}")
        start = time.time()

        try:
            result  = module(question=question)
            elapsed = round(time.time() - start, 2)
            success = result.error is None

            print(f"  SQL      : {result.sql_query}")
            print(f"  Resultado: {result.results if success else f'ERRO: {result.error}'}")
            print(f"  Tempo    : {elapsed}s | Sucesso: {'Sim' if success else 'Não'}")

            results_log.append({
                "question": question,
                "sql":      result.sql_query,
                "success":  success,
                "time":     elapsed,
                "results":  result.results,
                "error":    result.error,
            })

        except Exception as e:
            ensure_ollama_running()
            elapsed = round(time.time() - start, 2)
            print(f"  Falha na pergunta {i}: {e}")
            results_log.append({
                "question": question,
                "sql":      "N/A",
                "success":  False,
                "time":     elapsed,
                "results":  None,
                "error":    str(e),
            })
            continue

    return results_log

print("Funções de teste definidas")

In [ ]:
all_results = {}

for model in MODELS:
    try:
        all_results[model] = test_model(model, test_questions)
    except Exception as e:
        print(f"\nErro geral ao testar {model}: {e}")
        all_results[model] = []

print("\nTodos os modelos testados!")